# 06 — Plane Integration Demo

Цель ноутбука: показать подключение к Plane, список проектов, задачи, рекомендацию COMPASS AI и markdown-комментарий для Plane.

Запись комментария по умолчанию выключена.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

In [ ]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from src.integration.plane_client import PlaneClient

try:
    client = PlaneClient.from_env()
    print("Plane API health:", client.api_healthcheck())

    projects = client.list_projects()
    projects_df = pd.DataFrame(projects)

    display(projects_df[["id", "name", "identifier"]].head())
except Exception as exc:
    print("Plane is not available or .env is not configured:", exc)
    projects_df = pd.DataFrame()

In [ ]:
if not projects_df.empty:
    project_id = projects_df.iloc[0]["id"]
    work_items = client.list_work_items(project_id)
    work_items_df = pd.DataFrame(work_items)

    print("project_id:", project_id)
    display(work_items_df[["id", "name", "priority"]].head())
else:
    project_id = None
    work_items_df = pd.DataFrame()

In [ ]:
from src.agents.orchestrator import recommend_synthetic_task

recommendation = recommend_synthetic_task(
    "TASK-0001",
    mode="balanced_workload",
    top_k=3,
    use_llm=False,
)

display(pd.DataFrame(recommendation["top_candidates"]))

In [ ]:
from src.integration.plane_comment_formatter import format_plane_recommendation_comment

comment = format_plane_recommendation_comment(recommendation)
print(comment)

In [ ]:
WRITE_BACK = False

if WRITE_BACK and project_id and not work_items_df.empty:
    work_item_id = work_items_df.iloc[0]["id"]
    print("Would write comment to:", project_id, work_item_id)
else:
    print("WRITE_BACK disabled. This notebook is safe by default.")